In [1]:
dataset = "mnist_red30_incremental"
root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

In [2]:
import pickle
import pandas as pd
from river import forest, stream
from river import tree
from river import metrics
import os
import numpy as np
import sklearn.metrics as skm

In [3]:
def return_metrics():
    return {
        "accuracy": metrics.BalancedAccuracy(),
        "kappa": metrics.CohenKappa()
    }

def make_dir(path):
    if path is not None:
        if not os.path.isdir(path):
            os.makedirs(path)
            
def test_cl(cl_table, models, X_test, y_test):
    for m in models:
        for metric in ["accuracy", "kappa"]:
            cl_table[m][metric].append([])
        for X_test_task, y_test_task in zip(X_test, y_test):
            pred = []  
            for x_cl in X_test_task:
                x_cl = {f"feat{i}": x_cl[i] for i in range(len(x_cl))}
                y_hat = models[m].predict_one(x_cl)
                y_hat = 0 if y_hat is None else y_hat
                pred.append(y_hat)
            cl_table[m]["accuracy"][-1].append(skm.accuracy_score(y_test_task, pred))
            cl_table[m]["kappa"][-1].append(skm.cohen_kappa_score(y_test_task, pred))
    return cl_table

In [4]:
df = pd.read_csv(os.path.join(root, "datasets", f"{dataset}_train.csv"), nrows=1)
last_task = df["task"].iloc[0]
converters = {c: float for c in df.columns if "feat" in c}
converters["target"] = int
converters["task"] = int

data_stream = stream.iter_csv(
    os.path.join(root, "datasets", f"{dataset}_train.csv"),
    converters=converters,
    target="target"
)

models = {
    "arf": forest.ARFClassifier(leaf_prediction="nb"),
    "hat": tree.HoeffdingAdaptiveTreeClassifier(
        grace_period=100,
        delta=1e-5,
        leaf_prediction="nb",
        nb_threshold=10,
    )
}

perf = {
    m: {
        "total": return_metrics(),
        "concept": return_metrics()
    } for m in models
}

perf_values = {
    m: {
        "total": {"accuracy": [], "kappa": []},
        "concept": {"accuracy": [], "kappa": []}
    } for m in models
}
perf_values["drifts"] = []

predictions = {m:[] for m in models}


cl_table = {
    m: {
        metric: [] for metric in ["accuracy", "kappa"]
    } for m in models
} 

df_test = pd.read_csv(os.path.join(root, "datasets", f"{dataset}_test.csv"))

X_test = []
y_test = []
for task in df_test["task"].unique():
    df_task = df_test[df_test["task"]==task]
    X_test.append(df_task.iloc[:,:-2].values)
    y_test.append(list(df_task["target"]))

In [5]:
df = pd.read_csv(os.path.join(root, "datasets", f"{dataset}_train.csv"))

In [6]:
for idx, (x, y) in enumerate(data_stream):
    print(f"Prequential {idx+1}", end="\r")
    if last_task != x["task"]:
        print()
        print("DRIFT")
        for m in perf:
            perf[m]["concept"] = return_metrics()
        cl_table = test_cl(cl_table, models, X_test, y_test)
        perf_values["drifts"].append(idx*mb_size)
    last_task = x["task"]
    del x["task"]
    for m in models:
        pred = models[m].predict_one(x)
        pred = 0 if pred is None else pred
        predictions[m].append(pred)
        for method in ["total", "concept"]:
            for metric in perf[m][method]:
                perf[m][method][metric] = perf[m][method][metric].update(y, pred)
                perf_values[m][method][metric].append(perf[m][method][metric].get())
        models[m].learn_one(x, y)
cl_table = test_cl(cl_table, models, X_test, y_test)

Prequential 12311
DRIFT
Prequential 24081
DRIFT
Prequential 35021
DRIFT
Prequential 46821
DRIFT


In [7]:
make_dir(os.path.join(root, "performance", dataset))
with open(os.path.join(root, "performance", dataset, "performance.pkl"), "wb") as f:
    pickle.dump(perf_values, f)
with open(os.path.join(root, "performance", dataset, "cl_table.pkl"), "wb") as f:
    pickle.dump(cl_table, f)